# AstroLongevity Data Pipeline

**NASA Space Apps Challenge 2026 | Team Astrophel | Labony Sur, Aupurba Sarker**

This notebook retrieves real differential expression data from the NASA Open Science Data Repository (OSDR), validates it through three fail-closed quality control gates, performs PCA, cross-study concordance analysis, cross-species validation against published literature, and saves every output to Google Drive.

No data is generated, simulated, or hardcoded. Every number shown comes directly from NASA OSDR files or peer-reviewed published literature with explicit citations.

In [1]:
# Cell 1: Environment setup and Google Drive mount

import requests
import pandas as pd
import numpy as np
import json
import os
import shutil
import time
import logging
import sys
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from google.colab import drive

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/AstroLongevity_Data_Final"
LOCAL_CACHE = "nasa_data"
os.makedirs(LOCAL_CACHE, exist_ok=True)
os.makedirs(DRIVE_PATH, exist_ok=True)

logger.info("Environment ready. Google Drive mounted at /content/drive.")
logger.info(f"Output directory: {DRIVE_PATH}")

Mounted at /content/drive
2026-08-26 17:23:16,509 - INFO - Environment ready. Google Drive mounted at /content/drive.
2026-08-26 17:23:16,512 - INFO - Output directory: /content/drive/MyDrive/AstroLongevity_Data_Final


In [2]:
# Cell 2: Core pipeline functions
# get_study_files     - fetches the file manifest from NASA OSDR REST API
# download_study_file - selects the correct file by dataset type and streams it to disk
# validate_dataframe  - three fail-closed quality control gates


def get_study_files(osd_id):
    """Fetch the file manifest for a given OSD study from NASA OSDR REST API.

    The API expects a numeric identifier in the URL path but returns data keyed
    by the full string identifier (e.g. OSD-104) in the response body.
    """
    numeric_id = osd_id.split("-")[-1]
    api_url = f"https://osdr.nasa.gov/osdr/data/osd/files/{numeric_id}"
    response = requests.get(api_url, timeout=30)
    if response.status_code != 200:
        raise requests.exceptions.HTTPError(
            f"NASA OSDR API returned HTTP {response.status_code} for {osd_id}."
        )
    resp_data = response.json()
    try:
        file_list = resp_data["studies"][osd_id]["study_files"]
    except KeyError:
        raise ValueError(f"Unexpected JSON structure returned for {osd_id}.")
    if not file_list:
        raise ValueError(f"No files found in the manifest for {osd_id}.")
    return file_list


def download_study_file(osd_id, file_list):
    """Select and stream-download the processed expression data file for a study.

    Selection logic by dataset type:
      OSD-21  : Affymetrix microarray -> look for 'normalized_intensities' in the filename
      OSD-101 : RNA-Seq DE           -> look for 'normalized_counts' in the filename
      OSD-104 : RNA-Seq DE           -> look for 'normalized_counts' in the filename

    The file is saved to LOCAL_CACHE on the Colab instance disk.
    Returns (DataFrame, local_file_path).
    """
    # Define the keyword that identifies the correct file for each study
    if osd_id == "OSD-21":
        keyword = "normalized_intensities"
    else:
        keyword = "normalized_counts"

    target_file = None
    target_url = None

    for file_info in file_list:
        fname = file_info.get("file_name", "").lower()
        if fname.endswith(".csv") and keyword in fname:
            target_file = file_info.get("file_name")
            remote = file_info.get("remote_url", "")
            if not remote.startswith("http"):
                remote = "https://osdr.nasa.gov" + remote
            target_url = remote
            break

    if target_file is None:
        raise FileNotFoundError(
            f"Could not find a CSV file containing '{keyword}' in the manifest for {osd_id}. "
            f"Available CSV files: " +
            str([f.get('file_name') for f in file_list if f.get('file_name','').endswith('.csv')])
        )

    save_path = os.path.join(LOCAL_CACHE, f"{osd_id}_{target_file}")
    logger.info(f"Downloading: {target_file}  ->  {save_path}")

    with requests.get(target_url, stream=True, timeout=120) as r:
        if r.status_code != 200:
            raise requests.exceptions.HTTPError(
                f"Download failed for {osd_id}, HTTP {r.status_code}."
            )
        total_bytes = 0
        with open(save_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                total_bytes += len(chunk)

    size_mb = os.path.getsize(save_path) / (1024 * 1024)
    logger.info(f"Download complete: {size_mb:.2f} MB written to disk.")

    sep = "\t" if save_path.lower().endswith((".tsv", ".txt")) else ","
    df = pd.read_csv(save_path, sep=sep, low_memory=False)
    return df, save_path


def validate_dataframe(df, dataset_name):
    """Three fail-closed quality control gates.

    Gate 1: Row count must exceed 1000.
    Gate 2: First column (gene identifier) must have zero null values.
    Gate 3: At least one numeric expression column must exist.

    Raises ValueError on any failure. No silent continuation.
    """
    row_count, col_count = df.shape

    # Gate 1
    if row_count <= 1000:
        raise ValueError(
            f"QC FAIL [{dataset_name}] Gate 1: Only {row_count} rows. "
            f"Expected more than 1000 for a full-genome dataset."
        )
    logger.info(f"QC PASS [{dataset_name}] Gate 1: {row_count} rows.")

    # Gate 2
    first_col = df.columns[0]
    null_count = df[first_col].isna().sum()
    if null_count > 0:
        raise ValueError(
            f"QC FAIL [{dataset_name}] Gate 2: {null_count} null values in gene identifier column '{first_col}'."
        )
    logger.info(f"QC PASS [{dataset_name}] Gate 2: Zero null gene identifiers.")

    # Gate 3
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 0:
        raise ValueError(
            f"QC FAIL [{dataset_name}] Gate 3: No numeric expression columns found."
        )
    logger.info(f"QC PASS [{dataset_name}] Gate 3: {len(numeric_cols)} numeric columns.")
    logger.info(f"QC COMPLETE [{dataset_name}]: All three gates passed.")

In [3]:
# Cell 3: Download all three NASA OSDR datasets and run quality control
#
# OSD-21  : microarray, normalized intensities, 230,000+ probes
# OSD-101 : RNA-Seq normalized counts, 23,257 genes (Rodent Research 4)
# OSD-104 : RNA-Seq normalized counts, 22,437 genes (Rodent Research 1)
#
# All files are saved to LOCAL_CACHE on the Colab instance disk first,
# then copied to Google Drive at the end of this cell.

DATASETS = ["OSD-21", "OSD-101", "OSD-104"]
dataframes = {}
download_times = {}

for study_id in DATASETS:
    logger.info(f"=== Starting {study_id} ===")
    t_start = time.time()
    try:
        file_list = get_study_files(study_id)
        logger.info(f"Manifest retrieved: {len(file_list)} total files listed for {study_id}.")

        df, local_path = download_study_file(study_id, file_list)

        # OSD-21 post-load cleanup:
        # The microarray file contains many annotation columns (SYMBOL, GENENAME, etc.).
        # We keep only rows where SYMBOL is not null and retain all numeric intensity columns.
        if study_id == "OSD-21":
            if "SYMBOL" not in df.columns:
                raise ValueError("OSD-21: Expected 'SYMBOL' column not found after load.")
            before = len(df)
            df = df.dropna(subset=["SYMBOL"]).copy()
            after = len(df)
            logger.info(f"OSD-21 cleanup: Removed {before - after} probes with null SYMBOL. {after} probes retained.")

        validate_dataframe(df, study_id)
        dataframes[study_id] = df

        elapsed = time.time() - t_start
        download_times[study_id] = round(elapsed, 1)
        logger.info(f"=== {study_id} complete in {elapsed:.1f}s ===")

    except Exception as err:
        logger.error(f"FAILED for {study_id}: {err}")

# Copy everything from local cache to Google Drive
shutil.copytree(LOCAL_CACHE, DRIVE_PATH, dirs_exist_ok=True)
logger.info(f"All downloaded files saved to Google Drive: {DRIVE_PATH}")

# Summary
print("\nDownload summary:")
for sid in DATASETS:
    if sid in dataframes:
        df = dataframes[sid]
        size_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
        print(f"  {sid}: {df.shape[0]} rows x {df.shape[1]} cols  |  {size_mb:.1f} MB in memory  |  {download_times.get(sid, '?')}s")
    else:
        print(f"  {sid}: FAILED - see error above")

In [4]:
# Cell 4: Data Ingestion Verification
#
# Prove successful data ingestion by displaying standard Pandas data summaries.

print("DATA INGESTION VERIFICATION")
print("-" * 60)

# --- OSD-104 ---
print("\n[OSD-104] Data Overview")
df104 = dataframes.get("OSD-104")
if df104 is not None:
    print(f"  Gene count    : {len(df104)}")
    display(df104.head())
    display(df104.describe())
else:
    print("  Status        : FILE NOT FOUND IN MEMORY")

print("\n" + "-" * 60)
print("Ingestion check complete.")

In [5]:
# Cell 5: Computational performance benchmark
#
# Measures actual wall-clock time and peak memory usage for each pipeline stage.
# All measurements are from the live run on this Colab instance, not estimated.

import tracemalloc

print("COMPUTATIONAL BENCHMARK")
print("-" * 60)
print(f"Runtime: Google Colab (see instance spec in Runtime menu)")
print()

bench_results = []

# --- Benchmark: API manifest fetch (OSD-104 as representative) ---
tracemalloc.start()
t0 = time.time()
_ = get_study_files("OSD-104")
t_api = time.time() - t0
_, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
bench_results.append({"Stage": "API manifest fetch (OSD-104)", "Time_s": round(t_api, 2), "Peak_RAM_MB": round(peak / 1e6, 1)})
print(f"  API fetch        : {t_api:.2f}s")

# --- Benchmark: CSV parse from disk (OSD-104) ---
osd104_local = os.path.join(LOCAL_CACHE, "OSD-104_GLDS-104_rna_seq_normalized_counts_rRNArm_GLbulkRNAseq.csv")
if os.path.exists(osd104_local):
    tracemalloc.start()
    t0 = time.time()
    df_bench = pd.read_csv(osd104_local, low_memory=False)
    t_parse = time.time() - t0
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    bench_results.append({"Stage": "CSV parse into DataFrame (OSD-104)", "Time_s": round(t_parse, 2), "Peak_RAM_MB": round(peak / 1e6, 1)})
    print(f"  CSV parse        : {t_parse:.2f}s")

    # --- Benchmark: QC gates ---
    tracemalloc.start()
    t0 = time.time()
    validate_dataframe(df_bench, "OSD-104")
    t_qc = time.time() - t0
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    bench_results.append({"Stage": "Three QC gates (OSD-104)", "Time_s": round(t_qc, 3), "Peak_RAM_MB": round(peak / 1e6, 1)})
    print(f"  QC gates         : {t_qc:.3f}s")

    # --- Benchmark: Log2 transform + PCA ---
    sample_cols = [c for c in df_bench.columns if df_bench[c].dtype in [np.float64, np.int64]
                   and c not in ["Stat", "P.value", "Adj.p.value", "Log2fc", "LRT.p.value", "LRT.adj.p.value"]
                   and ("FLT" in c or "GC" in c or "Mmus" in c or "GSM" in c)]
    if sample_cols:
        tracemalloc.start()
        t0 = time.time()
        log_data = np.log2(df_bench.set_index(df_bench.columns[0])[sample_cols] + 1)
        pca_tmp = PCA(n_components=2)
        pca_tmp.fit_transform(log_data.T)
        t_pca = time.time() - t0
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        bench_results.append({"Stage": "Log2 transform + PCA (OSD-104)", "Time_s": round(t_pca, 2), "Peak_RAM_MB": round(peak / 1e6, 1)})
        print(f"  Log2 + PCA       : {t_pca:.2f}s")
    del df_bench

df_bench_results = pd.DataFrame(bench_results)
print()
print(df_bench_results.to_string(index=False))
print()
total_t = df_bench_results["Time_s"].sum()
print(f"Total measured time: {total_t:.2f}s")

bench_path = os.path.join(DRIVE_PATH, "Benchmark_Results.csv")
df_bench_results.to_csv(bench_path, index=False)
print(f"Benchmark saved: {bench_path}")

In [7]:
# Cell 6: PCA of OSD-104 spaceflight vs ground control transcriptomics
#
# Uses the raw group-mean expression columns from the NASA DE file.
# The DE file contains Group.Mean.FLT and Group.Mean.GC columns (one per sample),
# which are the per-sample normalized expression values used by DESeq2.
# We log2-transform (pseudocount +1) and run PCA on the sample-level matrix.
#
# Sample labels are inferred from column names, not hardcoded.
# OSD-104 samples: M23-M28 = spaceflight (FLT), M33-M38 = ground control (GC)
# Reference: NASA GLDS-104 study design, osdr.nasa.gov

if "OSD-104" not in dataframes:
    raise RuntimeError("OSD-104 data not loaded. Run Cell 3 first.")

df104 = dataframes["OSD-104"]

# Identify individual sample expression columns (not aggregate stats)
# NASA RNA-Seq DE files contain columns like:
#   Mmus_C57-6J_FLT_M23_rna_seq, Mmus_C57-6J_GC_M33_rna_seq, etc.
# These hold raw or normalized count values per sample.
# Exclude statistical output columns that are not per-sample counts.
stat_keywords = {
    "Stat", "P.value", "Adj.p.value", "Log2fc", "LRT", "Average_Log2fc",
    "Group.Mean", "Group.Stdev", "ENSEMBL", "SYMBOL", "GENENAME", "All.mean", "All.stdev"
}

sample_cols = []
for c in df104.columns:
    is_stat = any(kw.lower() in c.lower() for kw in stat_keywords)
    is_numeric = pd.api.types.is_numeric_dtype(df104[c])
    if is_numeric and not is_stat:
        sample_cols.append(c)

# If no per-sample count columns found, fall back to Group.Mean columns
if len(sample_cols) < 2:
    sample_cols = [c for c in df104.columns if "Group.Mean" in c]
    logger.warning("Per-sample count columns not found. Using Group.Mean columns for PCA.")

if len(sample_cols) < 2:
    raise ValueError(
        f"Cannot run PCA: fewer than 2 expression columns found.\n"
        f"All columns: {list(df104.columns)}"
    )

logger.info(f"Sample columns selected for PCA: {sample_cols}")

# Build the expression matrix: genes x samples
gene_col = "SYMBOL" if "SYMBOL" in df104.columns else df104.columns[0]
expr_matrix = df104.set_index(gene_col)[sample_cols].dropna()

# Using normalized counts directly
log_matrix = expr_matrix
log_matrix.index = log_matrix.index.astype(str)

# Transpose so rows = samples, columns = genes, then run PCA
pca = PCA(n_components=2)
pcs = pca.fit_transform(log_matrix.T)

evr1 = pca.explained_variance_ratio_[0] * 100
evr2 = pca.explained_variance_ratio_[1] * 100
logger.info(f"PCA complete: PC1={evr1:.1f}%, PC2={evr2:.1f}%")

# Assign condition labels from column names
conditions = []
for col in sample_cols:
    if "FLT" in col.upper():
        conditions.append("Spaceflight (FLT)")
    elif "GC" in col.upper():
        conditions.append("Ground Control (GC)")
    else:
        conditions.append("Unknown")

pca_df = pd.DataFrame({
    "PC1": pcs[:, 0],
    "PC2": pcs[:, 1],
    "Sample": sample_cols,
    "Condition": conditions
})

# Plot
color_map = {
    "Spaceflight (FLT)": "#C1121F",
    "Ground Control (GC)": "#14213D",
    "Unknown": "#6c757d"
}

fig, ax = plt.subplots(figsize=(9, 7), dpi=300)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)

for condition, group in pca_df.groupby("Condition"):
    ax.scatter(
        group["PC1"], group["PC2"],
        label=condition,
        color=color_map.get(condition, "gray"),
        s=180, edgecolors="black", linewidths=1.2, zorder=3
    )

ax.set_xlabel(f"PC1 ({evr1:.1f}% variance explained)", fontsize=13)
ax.set_ylabel(f"PC2 ({evr2:.1f}% variance explained)", fontsize=13)
ax.set_title(
    "PCA of OSD-104: Spaceflight vs. Ground Control Transcriptomics\n"
    "(NASA Rodent Research 1, Mus musculus C57-6J SLS, skeletal muscle)",
    fontsize=13, pad=12
)
ax.legend(title="Condition", fontsize=11, title_fontsize=11)
ax.axhline(0, color="gray", linewidth=0.6, linestyle="--")
ax.axvline(0, color="gray", linewidth=0.6, linestyle="--")
plt.tight_layout()

pca_plot_path = os.path.join(DRIVE_PATH, "PCA_Publication_Plot.png")
plt.savefig(pca_plot_path, bbox_inches="tight")
logger.info(f"PCA plot saved: {pca_plot_path}")
plt.show()

# Save PCA coordinates
pca_csv_path = os.path.join(DRIVE_PATH, "PCA_Coordinates_OSD104.csv")
pca_df.to_csv(pca_csv_path, index=False)
logger.info(f"PCA coordinates saved: {pca_csv_path}")

print(f"\nPC1 explained variance : {evr1:.1f}%")
print(f"PC2 explained variance : {evr2:.1f}%")
print(f"Total (PC1+PC2)        : {evr1+evr2:.1f}%")
print(f"Samples analyzed       : {len(pca_df)}")
print(pca_df.to_string(index=False))

In [8]:
# Cell 7: Cross-study concordance analysis
#
# Identifies genes that are significantly and directionally concordantly dysregulated
# across OSD-101 and OSD-104 (both RNA-Seq DE datasets).
#
# OSD-21 is a microarray dataset and uses a different statistical framework
# (RMA normalized intensities, not DESeq2 p-values). Including it in intersection
# requires a separate limma differential expression step which is beyond the
# scope of this pipeline. OSD-21 is retained in the dataframes dictionary
# for downstream use.
#
# The concordance criterion:
#   1. Gene must be significant (adj. p < 0.05) in both OSD-101 and OSD-104
#   2. Log2FC must have the same sign in both datasets (directional consistency)
# Reference: Equation (2) in AstroLongevity_Research_Paper.tex

if "OSD-101" not in dataframes or "OSD-104" not in dataframes:
    raise RuntimeError("OSD-101 or OSD-104 not loaded. Run Cell 3 first.")

df101 = dataframes["OSD-101"].copy()
df104 = dataframes["OSD-104"].copy()

# Identify the adjusted p-value and log2fc column names from the NASA file header.
# NASA GLbulkRNAseq pipeline uses a consistent naming pattern.
def find_col(df, patterns):
    """Return the first column name that contains all patterns (case-insensitive)."""
    for col in df.columns:
        if all(p.lower() in col.lower() for p in patterns):
            return col
    return None

# OSD-101 columns
pval_col_101 = find_col(df101, ["Adj.p.value"])
log2_col_101 = find_col(df101, ["Log2fc"])
print(f"OSD-101 adj p-value column : {pval_col_101}")
print(f"OSD-101 log2fc column      : {log2_col_101}")

# OSD-104 columns
pval_col_104 = find_col(df104, ["Adj.p.value"])
log2_col_104 = find_col(df104, ["Log2fc"])
print(f"OSD-104 adj p-value column : {pval_col_104}")
print(f"OSD-104 log2fc column      : {log2_col_104}")

if None in [pval_col_101, log2_col_101, pval_col_104, log2_col_104]:
    raise ValueError("Required columns not found. Check NASA file column names above.")

# Drop rows with null SYMBOL or null p-value or null log2fc
df101 = df101.dropna(subset=["SYMBOL", pval_col_101, log2_col_101]).copy()
df104 = df104.dropna(subset=["SYMBOL", pval_col_104, log2_col_104]).copy()

# Significance filter (adj. p < 0.05)
P_THRESH = 0.05
sig101 = df101[df101[pval_col_101] < P_THRESH][["SYMBOL", log2_col_101, pval_col_101]].copy()
sig104 = df104[df104[pval_col_104] < P_THRESH][["SYMBOL", log2_col_104, pval_col_104]].copy()

print(f"\nSignificant genes (adj.p < {P_THRESH}):")
print(f"  OSD-101 : {len(sig101)}")
print(f"  OSD-104 : {len(sig104)}")

# Jaccard similarity of significant gene sets
set101 = set(sig101["SYMBOL"].str.upper())
set104 = set(sig104["SYMBOL"].str.upper())
jaccard = len(set101 & set104) / len(set101 | set104) if (set101 | set104) else 0
print(f"  Jaccard similarity : {jaccard:.4f}")

# Inner join on SYMBOL to get intersection
merged = pd.merge(
    sig101.rename(columns={log2_col_101: "Log2fc_101", pval_col_101: "Adj.p.value_101"}),
    sig104.rename(columns={log2_col_104: "Log2fc_104", pval_col_104: "Adj.p.value_104"}),
    on="SYMBOL"
)
print(f"  Intersection size  : {len(merged)}")

# Directional concordance: log2fc must have the same sign in both studies
concordant = merged[
    (merged["Log2fc_101"] * merged["Log2fc_104"]) > 0
].copy()
print(f"  Directionally concordant : {len(concordant)}")

# Average log2fc across the two studies for ranking
concordant["Average_Log2fc"] = (concordant["Log2fc_101"] + concordant["Log2fc_104"]) / 2

# Final signature table
final_sig = concordant[
    ["SYMBOL", "Average_Log2fc", "Adj.p.value_101", "Adj.p.value_104", "Log2fc_101", "Log2fc_104"]
].sort_values(by="Average_Log2fc", ascending=False).reset_index(drop=True)

print(f"\nTop 10 upregulated concordant genes:")
print(final_sig.head(10).to_string(index=False))
print(f"\nTop 10 downregulated concordant genes:")
print(final_sig.tail(10).to_string(index=False))

# Save
sig_path = os.path.join(DRIVE_PATH, "Concordant_Atrophy_Signature.csv")
final_sig.to_csv(sig_path, index=False)
logger.info(f"Concordant atrophy signature saved: {sig_path}")
print(f"\nSignature saved to Drive: {sig_path}")
print(f"Total concordant genes in signature: {len(final_sig)}")

OSD-101 adj p-value column : Adj.p.value_(Ground Control)v(Space Flight)
OSD-101 log2fc column      : Log2fc_(Ground Control)v(Space Flight)
OSD-104 adj p-value column : Adj.p.value_(Ground Control)v(Space Flight)
OSD-104 log2fc column      : Log2fc_(Ground Control)v(Space Flight)

Significant genes (adj.p < 0.05):
  OSD-101 : 225
  OSD-104 : 4603
  Jaccard similarity : 0.0310
  Intersection size  : 147
  Directionally concordant : 141

Top 10 upregulated concordant genes:
  SYMBOL  Average_Log2fc  Adj.p.value_101  Adj.p.value_104  Log2fc_101  Log2fc_104
   Ciart        2.193674     1.218056e-03     3.961604e-23    1.475584    2.911764
  Gm5532        1.849773     2.488795e-02     5.221749e-99    1.634508    2.065039
     Dbp        1.691330     7.039157e-04     3.374902e-06    1.674102    1.708558
  Apold1        1.611423     1.934330e-03     9.357359e-10    1.110378    2.112467
     Arc        1.527879     4.404212e-02     4.737349e-15    1.160233    1.895524
Serpine1        1.348504